# Ejercicio práctico 1 en Google Colab: redes sociales y productividad

Cuaderno del módulo *Periodismo de precisión*. Repetimos el análisis exploratorio (EDA) que hicimos en Excel, ahora con Python, en dos partes:

1. **Variable por variable (análisis univariado):** qué valores toma cada variable, cuál es su centro, cuánto varía, qué forma tiene y si hay valores atípicos.
2. **Relaciones entre variables (análisis bivariado):** si existen correlaciones entre las variables cuantitativas y si la productividad cambia según los grupos.

La pregunta que guía el ejercicio es la misma:

> ¿Las personas que pasan más horas al día en redes sociales tienen una productividad real menor?

**Cómo usar este cuaderno.** Ejecute las celdas en orden, de arriba hacia abajo, con el botón ▶ o con `Shift + Enter`. No necesita instalar nada ni descargar el archivo: la primera celda lo lee desde el sitio del módulo.

**Recuerde:** los datos son **simulados**. Sirven para practicar el método, no para sacar conclusiones sobre personas reales.

## 0. Preparar el cuaderno y cargar los datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

URL = "https://raw.githubusercontent.com/nescolar/periodismo-de-precision/main/assets/datos/Dataset_redes_sociales_productividad_muestra500.xlsx"
df = pd.read_excel(URL, sheet_name="Datos")
df = df.drop(columns="fila_original")  # solo sirve para ubicar el registro en el archivo original
df.head()

### Paso 1. Reconocer la base

Antes de calcular cualquier cosa, respondemos tres preguntas: ¿cuántas filas y columnas hay?, ¿qué representa cada fila (la unidad de análisis)? y ¿de qué tipo es cada variable?

In [ ]:
print("Filas (personas):", df.shape[0])
print("Variables:", df.shape[1])
print("Valores faltantes en toda la base:", df.isna().sum().sum())
df.dtypes

Python distingue números (`int64`, `float64`) de texto (`str` u `object`), pero **no sabe** si un número es una escala ordinal. Eso lo decidimos nosotros. Clasificamos las variables a mano, siguiendo el diccionario de la hoja *Leeme*:

In [ ]:
cualitativas = ["genero", "sector_laboral", "red_preferida",
                "usa_apps_concentracion", "bienestar_digital_activo"]
ordinales = ["nivel_estres"]  # números del 1 al 10, pero son categorías ordenadas
cuantitativas = ["edad", "horas_redes_dia", "notificaciones_dia", "horas_trabajo_dia",
                 "productividad_percibida", "productividad_real", "horas_sueno",
                 "pantalla_antes_dormir", "pausas_trabajo", "cafes_dia",
                 "dias_agotamiento_mes", "horas_desconexion_semana", "satisfaccion_laboral"]

print(len(cualitativas), "cualitativas,", len(ordinales), "ordinal y", len(cuantitativas), "cuantitativas")

### Dos herramientas que usaremos para cada variable

Para no repetir el mismo código 19 veces, escribimos dos funciones: una para variables **cualitativas** y otra para **cuantitativas**. Luego las aplicamos a cada variable, una por una.

- `describir_cualitativa` muestra la **frecuencia** y el **porcentaje** de cada categoría, y un gráfico de barras ordenado de mayor a menor.
- `describir_cuantitativa` calcula la **media**, la **mediana**, la **desviación estándar**, los **cuartiles**, el **rango intercuartílico (RIC)**, la **asimetría** y los **valores atípicos** según la regla de 1,5 × RIC. Además dibuja un **histograma** y un **diagrama de caja**.

No hace falta entender cada línea de estas funciones; lo importante es saber leer lo que devuelven.

In [ ]:
def describir_cualitativa(col, orden=None):
    conteo = df[col].value_counts()
    if orden is not None:
        conteo = conteo.reindex(orden)
    tabla = pd.DataFrame({"Frecuencia": conteo,
                          "Porcentaje": (conteo / conteo.sum() * 100).round(1)})
    display(tabla)

    fig, ax = plt.subplots(figsize=(7, 0.45 * len(tabla) + 1.2))
    datos = tabla["Porcentaje"] if orden is not None else tabla["Porcentaje"].sort_values()
    ax.barh(datos.index.astype(str), datos.values, color="#3b6ea5")
    for i, v in enumerate(datos.values):
        ax.text(v + 0.5, i, f"{v:.1f} %", va="center", fontsize=9)
    ax.set_xlim(0, datos.max() * 1.18)
    ax.set_xlabel("% de personas")
    ax.set_title(f"{col} (n = {conteo.sum()})", loc="left")
    plt.tight_layout(); plt.show()


def describir_cuantitativa(col):
    s = df[col]
    q1, q3 = s.quantile([0.25, 0.75])
    ric = q3 - q1
    lim_inf, lim_sup = q1 - 1.5 * ric, q3 + 1.5 * ric
    atipicos = s[(s < lim_inf) | (s > lim_sup)]

    resumen = pd.Series({
        "n": s.count(), "Media": s.mean(), "Mediana": s.median(),
        "Desviación estándar": s.std(), "Mínimo": s.min(),
        "Q1 (25 %)": q1, "Q3 (75 %)": q3, "RIC": ric, "Máximo": s.max(),
        "Asimetría": s.skew(),
        "Límite inferior (1,5 × RIC)": lim_inf, "Límite superior (1,5 × RIC)": lim_sup,
        "Valores atípicos": len(atipicos),
    }, name=col)
    display(resumen.to_frame())

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.5), gridspec_kw={"width_ratios": [3, 1]})
    a1.hist(s, bins=20, color="#3b6ea5", edgecolor="white")
    a1.axvline(s.mean(), color="#c0392b", linestyle="--", label=f"Media = {s.mean():.2f}")
    a1.axvline(s.median(), color="#222222", linestyle=":", label=f"Mediana = {s.median():.2f}")
    a1.set_title(f"Histograma de {col}", loc="left"); a1.set_ylabel("Personas"); a1.legend(frameon=False)
    a2.boxplot(s, vert=True, widths=0.5)
    a2.set_xticks([]); a2.set_title("Diagrama de caja", loc="left")
    plt.tight_layout(); plt.show()

**Cómo leer los resultados de una variable cuantitativa**

- Si la **media** y la **mediana** están cerca, la distribución es más o menos simétrica. Si la media es bastante mayor que la mediana, hay una cola hacia la derecha (unos pocos valores muy altos).
- La **asimetría** cerca de 0 indica simetría; positiva, cola a la derecha; negativa, cola a la izquierda. Como guía, un valor mayor que 1 o menor que −1 es una asimetría marcada.
- La **desviación estándar** y el **RIC** dicen cuánto varían los datos. El RIC es el ancho del 50 % central y no se deja afectar por los extremos.
- Los **valores atípicos** son los que quedan fuera de los límites de 1,5 × RIC. Un atípico no es un error por definición: hay que revisarlo y decidir qué hacer con él.

## 1. Análisis univariado: variable por variable

### 1.1 Variables cualitativas

#### `genero`

Género declarado: Hombre, Mujer u Otro.

In [ ]:
describir_cualitativa("genero")

**Para comentar:** ¿Los grupos tienen tamaños parecidos? ¿Sería razonable comparar la productividad entre géneros con estos tamaños?

#### `sector_laboral`

Sector u ocupación de la persona (6 categorías).

In [ ]:
describir_cualitativa("sector_laboral")

**Para comentar:** ¿Qué sector tiene más personas y cuál menos? Si un sector tiene pocos casos, ¿qué cuidado hay que tener al compararlo?

#### `red_preferida`

Red social que la persona usa más (5 categorías).

In [ ]:
describir_cualitativa("red_preferida")

**Para comentar:** ¿Hay una red claramente dominante o están repartidas? ¿Es esto lo que esperaría en la vida real? Recuerde que los datos son simulados.

#### `usa_apps_concentracion`

Si la persona usa aplicaciones de concentración (Sí o No).

In [ ]:
describir_cualitativa("usa_apps_concentracion", orden=['Sí', 'No'])

**Para comentar:** ¿Qué porcentaje usa estas aplicaciones? Más adelante veremos si su productividad es distinta.

#### `bienestar_digital_activo`

Si la persona tiene activadas funciones de bienestar digital en su teléfono (Sí o No).

In [ ]:
describir_cualitativa("bienestar_digital_activo", orden=['Sí', 'No'])

**Para comentar:** ¿Se parece este porcentaje al de la variable anterior?

### 1.2 Variable ordinal: `nivel_estres`

Estrés autorreportado en una escala de 1 a 10. Aunque son números, son **categorías ordenadas**: la distancia entre 2 y 3 no tiene por qué ser igual a la distancia entre 8 y 9. Por eso la mostramos como frecuencias, en el orden de la escala, y le calculamos la **mediana**, que solo exige orden. La media la mostramos como referencia, pero conviene usarla con cuidado.

In [ ]:
describir_cualitativa("nivel_estres", orden=list(range(1, 11)))
print("Mediana:", df["nivel_estres"].median(), "| Moda:", df["nivel_estres"].mode()[0],
      "| Media (con cautela):", round(df["nivel_estres"].mean(), 2))

**Para comentar:** ¿Los niveles de estrés se concentran en algún punto de la escala o están repartidos por igual? ¿Qué diría eso sobre cómo se simularon los datos?

### 1.3 Variables cuantitativas

#### `edad`

Edad en años. Cuantitativa discreta.

In [ ]:
describir_cuantitativa("edad")

**Para comentar:** ¿Qué rango de edades cubre la muestra? ¿La distribución tiene alguna forma reconocible o es casi plana?

#### `horas_redes_dia`

Horas diarias en redes sociales. Cuantitativa continua. **Es una de las dos variables de la pregunta guía.**

In [ ]:
describir_cuantitativa("horas_redes_dia")

**Para comentar:** ¿Cuántas horas al día pasa en redes la persona típica? ¿Hay personas con valores atípicos? ¿Son creíbles?

#### `notificaciones_dia`

Notificaciones recibidas al día. Cuantitativa discreta.

In [ ]:
describir_cuantitativa("notificaciones_dia")

**Para comentar:** ¿La media y la mediana se parecen? ¿Qué rango de valores es el más común?

#### `horas_trabajo_dia`

Horas de trabajo al día. Cuantitativa continua.

In [ ]:
describir_cuantitativa("horas_trabajo_dia")

**Para comentar:** ¿Hay personas con 0 horas de trabajo? ¿Tiene sentido si pensamos en los sectores *Desempleado* y *Estudiante*?

#### `productividad_percibida`

Productividad que la persona cree tener, de 0 a 10. Cuantitativa continua.

In [ ]:
describir_cuantitativa("productividad_percibida")

**Para comentar:** Compare su media y su dispersión con las de `productividad_real`, que viene a continuación.

#### `productividad_real`

Productividad medida, de 0 a 10. Cuantitativa continua. **Es la otra variable de la pregunta guía.**

In [ ]:
describir_cuantitativa("productividad_real")

**Para comentar:** ¿La productividad real es, en promedio, mayor o menor que la percibida? ¿Cuál de las dos varía más?

#### `horas_sueno`

Horas de sueño por noche. Cuantitativa continua.

In [ ]:
describir_cuantitativa("horas_sueno")

**Para comentar:** ¿Cuánto duerme la persona típica? ¿Qué porcentaje aproximado duerme menos de 6 horas? (Pista: mire el histograma).

#### `pantalla_antes_dormir`

Horas frente a una pantalla antes de dormir. Cuantitativa continua.

In [ ]:
describir_cuantitativa("pantalla_antes_dormir")

**Para comentar:** ¿La distribución es simétrica o tiene una cola? ¿Qué medida de centro usaría para reportarla?

#### `pausas_trabajo`

Número de pausas durante la jornada. Cuantitativa discreta.

In [ ]:
describir_cuantitativa("pausas_trabajo")

**Para comentar:** ¿Los valores están repartidos por igual entre el mínimo y el máximo? ¿Qué forma tiene el histograma?

#### `cafes_dia`

Tazas de café al día. Cuantitativa discreta.

In [ ]:
describir_cuantitativa("cafes_dia")

**Para comentar:** ¿Cuál es el valor más frecuente? ¿Hay atípicos?

#### `dias_agotamiento_mes`

Días al mes en que la persona se siente agotada (*burnout*). Cuantitativa discreta.

In [ ]:
describir_cuantitativa("dias_agotamiento_mes")

**Para comentar:** ¿Cuántos días de agotamiento reporta la persona típica? ¿La distribución es simétrica?

#### `horas_desconexion_semana`

Horas semanales sin conexión a internet. Cuantitativa continua.

In [ ]:
describir_cuantitativa("horas_desconexion_semana")

**Para comentar:** ¿La media se aleja de la mediana? ¿Qué nos dice eso sobre la forma de la distribución?

#### `satisfaccion_laboral`

Satisfacción con el trabajo, de 0 a 10. Cuantitativa continua.

In [ ]:
describir_cuantitativa("satisfaccion_laboral")

**Para comentar:** ¿La satisfacción típica es alta, media o baja? ¿Hay personas en los extremos de la escala?

### 1.4 Resumen de todas las variables cuantitativas

La función `describe()` reúne en una sola tabla las medidas principales. Es la versión rápida de lo que hicimos variable por variable, y equivale a la hoja *Resumen* del Excel.

In [ ]:
resumen = df[cuantitativas + ordinales].describe().T
resumen["asimetría"] = df[cuantitativas + ordinales].skew()
resumen

### 1.5 Una variable nueva: la brecha de productividad

En el Excel creamos una columna con la diferencia entre la productividad que la persona **cree** tener y la que **realmente** tiene. Si es positiva, la persona se sobreestima; si es negativa, se subestima.

In [ ]:
df["brecha_productividad"] = df["productividad_percibida"] - df["productividad_real"]
describir_cuantitativa("brecha_productividad")
print("Porcentaje de personas que se sobreestiman:",
      round((df["brecha_productividad"] > 0).mean() * 100, 1), "%")

**Para comentar:** ¿La mayoría se sobreestima o se subestima? ¿Por cuánto, en promedio? ¿Sería esto un hallazgo noticioso si los datos fueran reales?

## 2. Relaciones entre variables: ¿existen correlaciones?

Ahora pasamos de describir cada variable por separado a mirar **cómo se mueven juntas**.

El **coeficiente de correlación de Pearson** (*r*) mide qué tan fuerte es la relación **lineal** entre dos variables cuantitativas. Va de −1 a 1:

| Valor de *r* | Cómo se lee |
|:--|:--|
| Cerca de 1 | Cuando una sube, la otra también sube |
| Cerca de −1 | Cuando una sube, la otra baja |
| Cerca de 0 | No hay relación lineal |

Como guía práctica (no es una regla fija): |*r*| < 0,1 es prácticamente nula; entre 0,1 y 0,3, débil; entre 0,3 y 0,5, moderada; y mayor que 0,5, fuerte.

Dos advertencias que un periodista de precisión no olvida:

1. **Correlación no es causalidad.** Que dos variables se muevan juntas no prueba que una cause la otra; puede haber una tercera variable detrás.
2. *r* **solo mide relaciones en línea recta.** Siempre conviene mirar el diagrama de dispersión.

### 2.1 Matriz de correlaciones

In [ ]:
corr = df[cuantitativas + ["brecha_productividad"]].corr(method="pearson")

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=60, ha="right", fontsize=9)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns, fontsize=9)
for i in range(len(corr)):
    for j in range(len(corr)):
        v = corr.iloc[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7,
                color="white" if abs(v) > 0.6 else "black")
fig.colorbar(im, ax=ax, shrink=0.7, label="r de Pearson")
ax.set_title("Matriz de correlaciones (Pearson)", loc="left")
plt.tight_layout(); plt.show()

**Cómo leer la matriz.** Cada celda es la correlación entre la variable de la fila y la de la columna. La diagonal siempre vale 1, porque es cada variable consigo misma. Los colores rojos indican correlación positiva; los azules, negativa; los blancos, casi nula. La matriz es simétrica: la parte de arriba repite la de abajo.

### 2.2 Las correlaciones más fuertes

In [ ]:
pares = (corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
             .stack().rename("r").reset_index())
pares.columns = ["Variable 1", "Variable 2", "r"]
pares["|r|"] = pares["r"].abs()

def fuerza(r):
    r = abs(r)
    return "Prácticamente nula" if r < 0.1 else "Débil" if r < 0.3 else "Moderada" if r < 0.5 else "Fuerte"

pares["Fuerza"] = pares["r"].apply(fuerza)
pares.sort_values("|r|", ascending=False).head(12).drop(columns="|r|").reset_index(drop=True)

**Para comentar:** ¿Cuáles son las relaciones más fuertes? ¿Alguna es obvia porque una variable se construyó a partir de otra? (Piense en `brecha_productividad`).

### 2.3 ¿Con qué se relaciona la productividad real?

Esta tabla ordena todas las variables según su correlación con `productividad_real`, que es la variable que nos interesa explicar.

In [ ]:
con_prod = corr["productividad_real"].drop("productividad_real").sort_values()
display(con_prod.to_frame("r con productividad_real").assign(Fuerza=con_prod.apply(fuerza)))

fig, ax = plt.subplots(figsize=(7, 6))
colores = ["#c0392b" if v > 0 else "#3b6ea5" for v in con_prod.values]
ax.barh(con_prod.index, con_prod.values, color=colores)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlim(-1, 1); ax.set_xlabel("r de Pearson")
ax.set_title("Correlación de cada variable con la productividad real", loc="left")
plt.tight_layout(); plt.show()

### 2.4 La pregunta guía: horas en redes y productividad real

Calculamos *r* y lo acompañamos del diagrama de dispersión, con una línea de tendencia.

In [ ]:
def dispersion(x, y):
    r = df[x].corr(df[y])
    m, b = np.polyfit(df[x], df[y], 1)
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.scatter(df[x], df[y], alpha=0.45, s=18, color="#3b6ea5")
    xs = np.linspace(df[x].min(), df[x].max(), 100)
    ax.plot(xs, m * xs + b, color="#c0392b", linewidth=2, label=f"Tendencia (r = {r:.2f})")
    ax.set_xlabel(x); ax.set_ylabel(y); ax.legend(frameon=False)
    ax.set_title(f"{y} según {x}", loc="left")
    plt.tight_layout(); plt.show()
    print(f"r = {r:.3f} ({fuerza(r).lower()}). Por cada unidad adicional de {x}, "
          f"{y} cambia en promedio {m:+.2f}.")

dispersion("horas_redes_dia", "productividad_real")

**Para comentar:** Con este resultado, ¿qué respondería a la pregunta guía? ¿La relación es tan fuerte como sugiere la percepción popular? Escriba un titular **honesto** de una línea.

### 2.5 Otras relaciones que vale la pena mirar

In [ ]:
dispersion("productividad_percibida", "productividad_real")

¿Las personas que se creen más productivas lo son realmente? ¿Qué tan cerca están los puntos de la línea?

In [ ]:
dispersion("horas_sueno", "productividad_real")

¿Dormir más se asocia con mayor productividad en estos datos?

In [ ]:
dispersion("horas_redes_dia", "horas_sueno")

¿Quienes pasan más horas en redes duermen menos? Si fuera así, ¿podría el sueño ser una **tercera variable** que conecta redes y productividad?

### 2.6 Una comprobación: correlación de Spearman

La correlación de **Spearman** usa el **orden** de los datos (sus rangos) en lugar de sus valores. Es útil cuando hay valores atípicos, cuando la relación es creciente pero no en línea recta, o cuando una variable es ordinal, como `nivel_estres`. Si Pearson y Spearman dan resultados parecidos, la conclusión es más sólida.

In [ ]:
variables_clave = ["horas_redes_dia", "notificaciones_dia", "horas_sueno", "nivel_estres",
                   "productividad_percibida", "satisfaccion_laboral", "dias_agotamiento_mes"]
comparacion = pd.DataFrame({
    "Pearson": df[variables_clave].corrwith(df["productividad_real"], method="pearson"),
    "Spearman": df[variables_clave].corrwith(df["productividad_real"], method="spearman"),
})
comparacion["Diferencia"] = comparacion["Spearman"] - comparacion["Pearson"]
comparacion.round(3)

### 2.7 Productividad según grupos (variables cualitativas)

La correlación de Pearson solo sirve entre variables cuantitativas. Para ver si la productividad cambia según una variable **cualitativa**, comparamos la **mediana** y la **media** de cada grupo y dibujamos diagramas de caja lado a lado.

In [ ]:
def por_grupos(grupo, y="productividad_real"):
    tabla = (df.groupby(grupo)[y].agg(n="count", Media="mean", Mediana="median", Desv_estándar="std")
               .sort_values("Mediana", ascending=False))
    display(tabla)
    orden = tabla.index.tolist()
    fig, ax = plt.subplots(figsize=(8, 0.5 * len(orden) + 1.5))
    ax.boxplot([df.loc[df[grupo] == g, y] for g in orden], vert=False, widths=0.5)
    ax.set_yticks(range(1, len(orden) + 1)); ax.set_yticklabels(orden)
    ax.invert_yaxis(); ax.set_xlabel(y)
    ax.set_title(f"{y} según {grupo}", loc="left")
    plt.tight_layout(); plt.show()

por_grupos("sector_laboral")

In [ ]:
por_grupos("usa_apps_concentracion")

In [ ]:
por_grupos("bienestar_digital_activo")

In [ ]:
por_grupos("red_preferida")

In [ ]:
por_grupos("genero")

**Para comentar:** ¿En qué variable cualitativa las cajas se ven más separadas? Si las cajas casi se superponen, la diferencia entre grupos es pequeña comparada con la variación dentro de cada grupo.

Para decir si una diferencia entre grupos es **estadísticamente significativa** necesitamos las herramientas de inferencia que veremos más adelante en el módulo. Por ahora, solo describimos.

### 2.8 Relación entre dos variables cualitativas: tabla cruzada

Por último, cruzamos dos variables cualitativas. Calculamos el porcentaje **dentro de cada fila**, para comparar grupos de distinto tamaño.

In [ ]:
cruce = pd.crosstab(df["sector_laboral"], df["usa_apps_concentracion"], normalize="index") * 100
cruce.round(1)

¿El uso de aplicaciones de concentración es parecido en todos los sectores, o hay alguno que se destaque?

## 3. Conclusiones

Antes de cerrar, escriba en la celda de abajo (haga doble clic para editarla):

1. **Tres hallazgos** del análisis univariado.
2. **La correlación más fuerte** que encontró y si tiene una explicación sencilla.
3. **La respuesta a la pregunta guía**, con el valor de *r*.
4. **Un titular honesto** que no exagere lo que dicen los datos ni afirme una causa.
5. **Una limitación** del análisis (recuerde: los datos son simulados, y en la muestra solo quedaron los registros sin valores faltantes).

*Escriba aquí sus conclusiones.*

1.
2.
3.
4.
5.